### Imports

In [ ]:
import img2pdf

from kcatbench import plotting
from kcatbench.util import RESULT_DIR, read_csv_with_schema
from kcatbench.dataset.util import aggregate_rows_by_columns

### Data preparation

Load dataset with predictions

In [ ]:
df = read_csv_with_schema(RESULT_DIR / "predictions" / "your_prediction_result.csv")

Aggregate duplicate rows based on model input. For example based on "substrates" and "sequence".

In [ ]:
df_agg = aggregate_rows_by_columns(df, key_columns=["substrates", "sequence"], aggregation_strategies={"experimental_kcat": "max", "dlkcat_kcat": "mean", "unikp_kcat": "mean", "turnup_kcat": "mean", "catpred_kcat": "mean", "catapro_kcat": "mean", "mmkcat_kcat": "mean",})

Then create a consensus dataset with only inputs that each model could predict.

In [ ]:
df_agg_consensus = df_agg.dropna(subset=["dlkcat_kcat", "unikp_kcat", "turnup_kcat", "catpred_kcat", "catapro_kcat", "mmkcat_kcat"])

### Plotting

In [ ]:
model_names = {
    'dlkcat_kcat': 'DLKcat', 
    'catapro_kcat': 'CataPro', 
    'turnup_kcat': 'TurNuP', 
    'unikp_kcat': 'UniKP', 
    'catpred_kcat': 'CatPred', 
    'mmkcat_kcat': 'MMKcat'
}

In [ ]:
plotting.plot_model_intersection_sets(
    df_agg_consensus,
    model_names,
    subset_type='best',
    percentage=10,
    save=True,
    show=False
)

In [ ]:
plotting.plot_model_intersection_sets(
    df_agg_consensus,
    model_names,
    subset_type='worst',
    percentage=10,
    save=True,
    show=False
)

In [ ]:
model_items = list(model_names.items())

for i, (model1_col, model1_name) in enumerate(model_items):
    plotting.plot_model_comparison(
        df_agg_consensus,
        model1_col,
        "experimental_kcat",
        model1_name,
        "Experimental",
        log_scale=True,
        save=True,
        gridsize=70,
        show=False,
        show_ellipse_stats=True,
        ellipse_stats_position="lower_right",
    )

    for model2_col, model2_name in model_items[i + 1:]:
        plotting.plot_model_comparison(
            df_agg_consensus,
            model1_col,
            model2_col,
            model1_name,
            model2_name,
            log_scale=True,
            save=True,
            gridsize=70,
            show=False,
            model_vs_model=True,
        )

Create a pdf with all comparison plots generated in the previous cell.

In [ ]:
plot_dir = RESULT_DIR / "plots" / "comparison_plots"

files = sorted([str(f) for f in plot_dir.glob("*.png")])

with open(str(plot_dir / "all_plots.pdf"), "wb") as f:
    f.write(img2pdf.convert(files))